# Notebook 02 - Staging, parsing et standardisation

## Objectif

Dans ce notebook, on passe de la couche brute à une couche de staging lisible et contrôlée.

On va construire trois niveaux :

- `stg_raw_notice` : une vue brute aplatie minimale
- `stg_notice_std` : les lignes valides et standardisées
- `stg_reject_notice` : les lignes rejetées
- `stg_notice_cpv` : la collection répétable des CPV

## Principe important

On ne met **pas** encore les données dans le DWH.  
On prépare seulement une donnée propre, typée et contrôlée.

In [1]:
# Nous importons json pour relire les payloads bruts.
import json

# Nous importons pandas pour les manipulations tabulaires.
import pandas as pd

# Nous importons la bibliothèque pathlib pour manipuler les chemins de fichiers proprement.
from pathlib import Path

In [2]:
# Nous définissons un chemin de départ qui correspond au dossier courant du notebook.
current_dir = Path.cwd().resolve()

# Si le notebook est exécuté depuis le dossier du projet, ce test sera vrai.
if (current_dir / "data").exists():
    # Dans ce cas, le dossier courant est déjà la racine du projet.
    project_root = current_dir
else:
    # Sinon, on suppose que le notebook est exécuté depuis le dossier notebooks.
    project_root = current_dir.parent

# Nous définissons ici le dossier qui contient les lots bruts.
data_dir = project_root / "data" / "raw_batches"

# Nous définissons ici le dossier de sortie.
output_dir = project_root / "outputs"

# Nous affichons la racine retenue pour éviter toute ambiguïté.
print("project_root =", project_root)

# Nous affichons la liste des fichiers bruts disponibles.
print("raw files =", sorted(p.name for p in data_dir.glob("*.json")))

project_root = D:\Ens-EPSI\boamp_etl_dwh_tiny_explained_project
raw files = ['B20150308_01_initial.json', 'B20150309_01_incremental.json', 'B20150309_01_rerun.json']


## Étape 1 - Charger la couche brute produite par le notebook précédent

In [3]:
# Nous lisons la table brute `raw_notice`.
raw_notice = pd.read_csv(output_dir / "staging" / "raw_notice.csv")

# Nous convertissons `extraction_ts` en timestamp UTC.
raw_notice["extraction_ts"] = pd.to_datetime(raw_notice["extraction_ts"], utc=True)

# Nous affichons les premières lignes.
raw_notice.head()

,batch_id,batch_type,source_system,extraction_ts,page_no,record_hash,payload_json
0,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,1,510aba836dde6afdee6d4a1d83a3747b5c89ebd2e6c4f0...,"{""code_departement"": [""35""], ""code_departement..."
1,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,2,283b0d32917fa437e5761988a87d21242e8c0fefca0203...,"{""code_departement"": [""35""], ""code_departement..."
2,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,3,13848b9951b0faafb1c5dc92917c9e15982abbc6599191...,"{""code_departement"": [""44""], ""code_departement..."
3,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,4,66674b9826d0cf1f35b23687d7ec4d9fab1cd75c0c1bdb...,"{""code_departement"": [""44""], ""code_departement..."
4,B20150309_01_incremental,INCREMENTAL,BOAMP_API,2015-03-09 05:30:00+00:00,1,3bb0e6ed1ffaad43668165435dd93e592aed0402179744...,"{""code_departement"": [""69""], ""code_departement..."


## Étape 2 - Définir de petites fonctions utilitaires

In [4]:
# ----------------------------------------------------------------------------------------
# Nous définissons une fonction pour parser un JSON qui peut déjà être un dictionnaire,
# être une chaîne JSON, ou être vide.
# ----------------------------------------------------------------------------------------

def parse_json_text(value):
    """
    Convertir une valeur potentiellement JSON en dictionnaire Python.

    Paramètres
    ----------
    value : any
        Valeur potentiellement vide, dictionnaire ou chaîne JSON.

    Retour
    ------
    dict
        Dictionnaire Python. Si la valeur est vide, on retourne un dictionnaire vide.
    """

    # Si la valeur est déjà un dictionnaire, on la retourne telle quelle.
    if isinstance(value, dict):
        return value

    # Si la valeur est absente, on retourne un dictionnaire vide.
    if value is None:
        return {}

    # Nous convertissons la valeur en texte propre.
    text = str(value).strip()

    # Si le texte est vide, on retourne un dictionnaire vide.
    if text == "":
        return {}

    # Nous interprétons le texte comme du JSON.
    return json.loads(text)


# ----------------------------------------------------------------------------------------
# Nous définissons une fonction de nettoyage simple pour les chaînes.
# ----------------------------------------------------------------------------------------

def normalize_text(value):
    """
    Nettoyer une valeur textuelle simple.

    Paramètres
    ----------
    value : any
        Valeur brute.

    Retour
    ------
    str | None
        Texte nettoyé ou None si la valeur est vide.
    """

    # Si la valeur est absente, on retourne None.
    if value is None:
        return None

    # Nous transformons la valeur en texte puis nous supprimons les espaces inutiles.
    text = str(value).strip()

    # Si le texte est vide ou vaut NULL, on retourne None.
    if text == "" or text.upper() == "NULL":
        return None

    # Sinon on retourne le texte nettoyé.
    return text


# ----------------------------------------------------------------------------------------
# Nous définissons une fonction robuste de parsing de date.
# ----------------------------------------------------------------------------------------

def parse_datetime_safe(value):
    """
    Convertir une valeur en timestamp pandas UTC.

    Paramètres
    ----------
    value : any
        Valeur brute.

    Retour
    ------
    pandas.Timestamp | pandas.NaT
        Timestamp UTC ou valeur manquante si le parsing échoue.
    """

    # Nous encapsulons le parsing dans un try pour éviter qu'une mauvaise date casse tout le pipeline.
    try:

        # Si la valeur est vide, nous retournons NaT.
        if value in (None, "", "NULL"):
            return pd.NaT

        # Nous utilisons pandas pour parser la date en UTC.
        return pd.to_datetime(value, utc=True)

    # En cas d'erreur, nous retournons NaT.
    except Exception:
        return pd.NaT

## Étape 3 - Construire `stg_raw_notice`

On aplatit juste le minimum utile :

- l'identifiant BOAMP,
- l'objet,
- les dates brutes,
- l'acheteur,
- le type de marché,
- le contenu `gestion`,
- le contenu `donnees`,
- les CPV.

In [5]:
# Nous créons une liste vide pour les lignes aplaties.
stg_raw_rows = []

# Nous parcourons toutes les lignes de la table brute.
for raw_row in raw_notice.to_dict("records"):

    # Nous rechargeons le payload JSON complet.
    record = json.loads(raw_row["payload_json"])

    # Nous parsions le bloc `gestion`.
    gestion = parse_json_text(record.get("gestion"))

    # Nous parsions le bloc `donnees`.
    donnees = parse_json_text(record.get("donnees"))

    # Nous récupérons le type de marché top-level.
    type_marche_value = record.get("type_marche")

    # Si `type_marche` est une liste, on prend le premier élément.
    if isinstance(type_marche_value, list):
        type_marche_top = type_marche_value[0] if type_marche_value else None
    else:
        type_marche_top = type_marche_value

    # Nous ajoutons une ligne aplatie minimale.
    stg_raw_rows.append(
        {
            # Colonnes de traçabilité.
            "batch_id": raw_row["batch_id"],
            "batch_type": raw_row["batch_type"],
            "source_system": raw_row["source_system"],
            "extraction_ts": raw_row["extraction_ts"],
            "record_hash": raw_row["record_hash"],

            # Colonnes top-level utiles.
            "idweb": normalize_text(record.get("idweb")),
            "objet": normalize_text(record.get("objet")),
            "dateparution_raw": normalize_text(record.get("dateparution")),
            "datelimitereponse_raw": normalize_text(record.get("datelimitereponse")),
            "nomacheteur_top": normalize_text(record.get("nomacheteur")),
            "type_marche_top": normalize_text(type_marche_top),
            "url_avis": normalize_text(record.get("url_avis")),

            # Colonnes utiles récupérées dans les conteneurs source.
            "gestion_publication_date": normalize_text(gestion.get("INDEXATION", {}).get("DATE_PUBLICATION")),
            "gestion_deadline_date": normalize_text(gestion.get("INDEXATION", {}).get("DATE_LIMITE_REPONSE")),
            "buyer_name_donnees": normalize_text(donnees.get("IDENTITE", {}).get("DENOMINATION")),

            # Nous conservons aussi la liste CPV brute pour l'étape suivante.
            "cpv_raw": donnees.get("OBJET", {}).get("CPV", []),
        }
    )

# Nous convertissons la liste en DataFrame.
stg_raw_notice = pd.DataFrame(stg_raw_rows)

# Nous affichons le résultat.
stg_raw_notice.head()

,batch_id,batch_type,source_system,extraction_ts,record_hash,idweb,objet,dateparution_raw,datelimitereponse_raw,nomacheteur_top,type_marche_top,url_avis,gestion_publication_date,gestion_deadline_date,buyer_name_donnees,cpv_raw
0,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,510aba836dde6afdee6d4a1d83a3747b5c89ebd2e6c4f0...,15-31501,concession d'aménagement pour la réalisation d...,2015-03-03,2015-04-20T11:00:00+00:00,Cté Cnes Moyenne Vilaine et Semnon,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-31501,2015-03-03,2015-04-20T12:00:00+01:00,Cté Cnes Moyenne Vilaine et Semnon,"[{'PRINCIPAL': '45111291'}, {'PRINCIPAL': '710..."
1,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,283b0d32917fa437e5761988a87d21242e8c0fefca0203...,15-40000,maintenance de bâtiments scolaires et travaux ...,2015-03-04,2015-03-18T12:00:00+00:00,Ville de Rennes,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40000,2015-03-04,2015-03-18T12:00:00+00:00,Ville de Rennes,"[{'PRINCIPAL': '45453100'}, {'SECONDAIRE': '50..."
2,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,13848b9951b0faafb1c5dc92917c9e15982abbc6599191...,15-40001,fourniture de matériel informatique pour les s...,2015-03-05,2015-03-25T16:00:00+00:00,Ville de Nantes,FOURNITURES,https://www.boamp.fr/pages/avis/?q=idweb:15-40001,2015-03-05,2015-03-25T16:00:00+00:00,Ville de Nantes,"[{'PRINCIPAL': '30200000'}, {'SECONDAIRE': '48..."
3,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,66674b9826d0cf1f35b23687d7ec4d9fab1cd75c0c1bdb...,15-40002,prestations de nettoyage des locaux administra...,2015-03-06,2015-03-20T10:00:00+00:00,Conseil départemental de Loire-Atlantique,SERVICES,https://www.boamp.fr/pages/avis/?q=idweb:15-40002,2015-03-06,2015-03-20T10:00:00+00:00,Conseil départemental de Loire-Atlantique,[{'PRINCIPAL': '90910000'}]
4,B20150309_01_incremental,INCREMENTAL,BOAMP_API,2015-03-09 05:30:00+00:00,3bb0e6ed1ffaad43668165435dd93e592aed0402179744...,15-40003,travaux d'aménagement de voirie et signalisation,2015-03-07,2015-03-30T15:00:00+00:00,Métropole de Lyon,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40003,2015-03-07,2015-03-30T15:00:00+00:00,Métropole de Lyon,"[{'PRINCIPAL': '45233140'}, {'SECONDAIRE': '34..."


## Étape 4 - Construire `stg_notice_std`, `stg_reject_notice` et `stg_notice_cpv`

Ici, on applique des règles de qualité.

Règles retenues :

- `idweb` obligatoire,
- date de publication valide,
- acheteur non vide,
- type de marché non vide,
- `deadline >= publication` si la deadline existe.

In [6]:
# Nous créons trois listes vides pour les trois sorties principales.
std_rows = []
reject_rows = []
cpv_rows = []

# Nous parcourons les lignes aplaties une par une.
for row in stg_raw_notice.to_dict("records"):

    # Nous choisissons une source de vérité simple pour la date de publication.
    publication_ts = parse_datetime_safe(row["dateparution_raw"] or row["gestion_publication_date"])

    # Nous choisissons une source de vérité simple pour la date limite.
    deadline_ts = parse_datetime_safe(row["datelimitereponse_raw"] or row["gestion_deadline_date"])

    # Nous choisissons l'acheteur top-level, sinon celui du bloc `donnees`.
    buyer_name = normalize_text(row["nomacheteur_top"]) or normalize_text(row["buyer_name_donnees"])

    # Nous normalisons le type de marché.
    market_type_code = normalize_text(row["type_marche_top"])

    # Nous créons une liste locale de motifs de rejet.
    reject_rules = []

    # Nous appliquons la règle R01.
    if not normalize_text(row["idweb"]):
        reject_rules.append("R01 idweb manquant")

    # Nous appliquons la règle R02.
    if pd.isna(publication_ts):
        reject_rules.append("R02 dateparution invalide")

    # Nous appliquons la règle R03.
    if not buyer_name:
        reject_rules.append("R03 acheteur manquant")

    # Nous appliquons la règle R04.
    if not market_type_code:
        reject_rules.append("R04 type_marche manquant")

    # Nous appliquons la règle R05.
    if not pd.isna(publication_ts) and not pd.isna(deadline_ts) and deadline_ts < publication_ts:
        reject_rules.append("R05 deadline avant publication")

    # Si au moins une règle est violée, la ligne part en quarantaine.
    if reject_rules:

        # Nous enregistrons la ligne rejetée avec son motif.
        reject_rows.append(
            {
                "batch_id": row["batch_id"],
                "idweb": row["idweb"],
                "reject_rule": " | ".join(reject_rules),
                "reject_reason": "Ligne mise en quarantaine",
                "raw_payload_hash": row["record_hash"],
            }
        )

        # Nous passons directement à la ligne suivante.
        continue

    # Nous déterminons si la ligne possède une deadline.
    has_deadline_flag = not pd.isna(deadline_ts)

    # Nous initialisons le délai à None.
    response_delay_days = None

    # Si la deadline existe, nous calculons le délai de réponse en jours.
    if has_deadline_flag:
        response_delay_days = int((deadline_ts.normalize() - publication_ts.normalize()).days)

    # Nous ajoutons la ligne standardisée valide.
    std_rows.append(
        {
            "batch_id": row["batch_id"],
            "record_hash": row["record_hash"],
            "idweb": row["idweb"],
            "objet": row["objet"],
            "buyer_name": buyer_name,
            "publication_ts": publication_ts,
            "deadline_ts": deadline_ts,
            "response_delay_days": response_delay_days,
            "market_type_code": market_type_code,
            "url_avis": row["url_avis"],
            "has_deadline_flag": has_deadline_flag,
        }
    )

    # Nous traitons maintenant la collection répétable des CPV.
    cpv_list = row["cpv_raw"] or []

    # Nous parcourons les CPV en gardant leur position.
    for index, cpv_item in enumerate(cpv_list, start=1):

        # Nous récupérons le code CPV principal ou secondaire.
        cpv_code = normalize_text(cpv_item.get("PRINCIPAL") or cpv_item.get("SECONDAIRE"))

        # Si le code CPV existe, nous l'enregistrons.
        if cpv_code:
            cpv_rows.append(
                {
                    "idweb": row["idweb"],
                    "cpv_code": cpv_code,
                    "is_primary_flag": index == 1,
                }
            )

# Nous créons les DataFrames finaux de staging.
stg_notice_std = pd.DataFrame(std_rows).sort_values(["publication_ts", "idweb"]).reset_index(drop=True)
stg_reject_notice = pd.DataFrame(reject_rows)
stg_notice_cpv = pd.DataFrame(cpv_rows).drop_duplicates().reset_index(drop=True)

# Nous affichons les trois sorties.
print("stg_notice_std")
display(stg_notice_std)

print("stg_reject_notice")
display(stg_reject_notice)

print("stg_notice_cpv")
display(stg_notice_cpv)


stg_notice_std


,batch_id,record_hash,idweb,objet,buyer_name,publication_ts,deadline_ts,response_delay_days,market_type_code,url_avis,has_deadline_flag
0,B20150308_01_initial,510aba836dde6afdee6d4a1d83a3747b5c89ebd2e6c4f0...,15-31501,concession d'aménagement pour la réalisation d...,Cté Cnes Moyenne Vilaine et Semnon,2015-03-03 00:00:00+00:00,2015-04-20 11:00:00+00:00,48,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-31501,True
1,B20150308_01_initial,283b0d32917fa437e5761988a87d21242e8c0fefca0203...,15-40000,maintenance de bâtiments scolaires et travaux ...,Ville de Rennes,2015-03-04 00:00:00+00:00,2015-03-18 12:00:00+00:00,14,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40000,True
2,B20150308_01_initial,13848b9951b0faafb1c5dc92917c9e15982abbc6599191...,15-40001,fourniture de matériel informatique pour les s...,Ville de Nantes,2015-03-05 00:00:00+00:00,2015-03-25 16:00:00+00:00,20,FOURNITURES,https://www.boamp.fr/pages/avis/?q=idweb:15-40001,True
3,B20150308_01_initial,66674b9826d0cf1f35b23687d7ec4d9fab1cd75c0c1bdb...,15-40002,prestations de nettoyage des locaux administra...,Conseil départemental de Loire-Atlantique,2015-03-06 00:00:00+00:00,2015-03-20 10:00:00+00:00,14,SERVICES,https://www.boamp.fr/pages/avis/?q=idweb:15-40002,True
4,B20150309_01_incremental,3bb0e6ed1ffaad43668165435dd93e592aed0402179744...,15-40003,travaux d'aménagement de voirie et signalisation,Métropole de Lyon,2015-03-07 00:00:00+00:00,2015-03-30 15:00:00+00:00,23,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40003,True
5,B20150309_01_rerun,3bb0e6ed1ffaad43668165435dd93e592aed0402179744...,15-40003,travaux d'aménagement de voirie et signalisation,Métropole de Lyon,2015-03-07 00:00:00+00:00,2015-03-30 15:00:00+00:00,23,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40003,True
6,B20150309_01_incremental,1183366902af82c4fce3ae680bbe556987e3d1100b9f5c...,15-40004,services d'assistance juridique en urbanisme,Communauté urbaine de Bordeaux,2015-03-08 00:00:00+00:00,2015-03-22 09:00:00+00:00,14,SERVICES,https://www.boamp.fr/pages/avis/?q=idweb:15-40004,True
7,B20150309_01_rerun,1183366902af82c4fce3ae680bbe556987e3d1100b9f5c...,15-40004,services d'assistance juridique en urbanisme,Communauté urbaine de Bordeaux,2015-03-08 00:00:00+00:00,2015-03-22 09:00:00+00:00,14,SERVICES,https://www.boamp.fr/pages/avis/?q=idweb:15-40004,True


stg_reject_notice


,batch_id,idweb,reject_rule,reject_reason,raw_payload_hash
0,B20150309_01_incremental,15-40005,R05 deadline avant publication,Ligne mise en quarantaine,298ad4fe740d8c713217c28544e7999d81c69f50ff18b6...
1,B20150309_01_rerun,15-40005,R05 deadline avant publication,Ligne mise en quarantaine,298ad4fe740d8c713217c28544e7999d81c69f50ff18b6...


stg_notice_cpv


,idweb,cpv_code,is_primary_flag
0,15-31501,45111291,True
1,15-31501,71000000,False
2,15-31501,70122100,False
3,15-40000,45453100,True
4,15-40000,50700000,False
5,15-40001,30200000,True
6,15-40001,48000000,False
7,15-40002,90910000,True
8,15-40003,45233140,True
9,15-40003,34992200,False


## Étape 5 - Sauvegarder les sorties de staging

In [7]:
# Nous définissons le dossier de sortie de staging.
staging_dir = output_dir / "staging"

# Nous créons le dossier si besoin.
staging_dir.mkdir(parents=True, exist_ok=True)

# Nous écrivons la vue brute aplatie.
stg_raw_notice.to_csv(staging_dir / "stg_raw_notice.csv", index=False)

# Nous écrivons les lignes standardisées valides.
stg_notice_std.to_csv(staging_dir / "stg_notice_std.csv", index=False)

# Nous écrivons la table des CPV.
stg_notice_cpv.to_csv(staging_dir / "stg_notice_cpv.csv", index=False)

# Nous écrivons la table de rejet.
stg_reject_notice.to_csv(staging_dir / "stg_reject_notice.csv", index=False)

# Nous affichons les fichiers produits.
sorted(p.name for p in staging_dir.glob("*.csv"))

['ops_batch_control.csv',
 'raw_notice.csv',
 'stg_notice_cpv.csv',
 'stg_notice_std.csv',
 'stg_raw_notice.csv',
 'stg_reject_notice.csv']

À la fin du staging :

- on ne mélange plus brut et standardisé,
- les lignes invalides ne disparaissent pas,
- les CPV sont isolés dans une table dédiée,
- les dates sont typées,
- les règles de rejet sont explicites.

> standardiser, contrôler, tracer, puis charger. 
